# 01 — Exploratory Data Analysis
SmartHire Phase 1: explore the raw Resume Dataset and the merged Job Corpus before modeling.

**Prerequisite:** run `python -m src.data.load_data` and `python -m src.data.preprocess` from the repo root first (or run the equivalent cells below) so `data/processed/*.csv` exist.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from src.config import RESUME_DATASET_CLEAN, JOB_CORPUS_CLEAN, FIGURES_DIR

sns.set_style('whitegrid')
pd.set_option('display.max_colwidth', 120)

## 1. Load cleaned datasets
(Run `src/data/load_data.py` + `src/data/preprocess.py` first if these files don't exist yet.)

In [ ]:
resumes = pd.read_csv(RESUME_DATASET_CLEAN)
jobs = pd.read_csv(JOB_CORPUS_CLEAN)

print('Resumes:', resumes.shape)
print('Jobs:', jobs.shape)
resumes.head()

## 2. Resume dataset — category distribution
Are the 25 categories balanced? This matters for the classifier (class_weight='balanced' is already set in `src/models/classifier.py`, but it's worth visualizing).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
order = resumes['category'].value_counts().index
sns.countplot(data=resumes, y='category', order=order, ax=ax)
ax.set_title('Resume Category Distribution')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'resume_category_distribution.png', dpi=150)
plt.show()

## 3. Resume text length distribution

In [ ]:
resumes['word_count'] = resumes['clean_text'].str.split().apply(len)

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(resumes['word_count'], bins=40, ax=ax)
ax.set_title('Resume Length (words) After Cleaning')
ax.set_xlabel('Word count')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'resume_word_count_distribution.png', dpi=150)
plt.show()

resumes['word_count'].describe()

## 4. Word cloud — most frequent terms per a sample category
Pick any category present in your data to sanity-check that cleaning kept meaningful tokens.

In [ ]:
sample_category = resumes['category'].value_counts().index[0]
text_blob = ' '.join(resumes.loc[resumes['category'] == sample_category, 'clean_text'])

wc = WordCloud(width=900, height=450, background_color='white').generate(text_blob)
plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title(f'Most Frequent Terms — {sample_category}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'wordcloud_{sample_category.replace(" ", "_").lower()}.png', dpi=150)
plt.show()

## 5. Job corpus — composition by source

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
jobs['source'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Job Postings by Source')
ax.set_ylabel('Count')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'job_corpus_by_source.png', dpi=150)
plt.show()

## 6. Job corpus — top locations and title keywords

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
jobs['location'].value_counts().head(15).plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_title('Top 15 Job Locations')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'job_corpus_top_locations.png', dpi=150)
plt.show()

In [ ]:
jobs['title_word_count'] = jobs['clean_text'].str.split().apply(len)
jobs[['title', 'company', 'location', 'source', 'title_word_count']].sample(10, random_state=42)

## 7. Notes / findings (fill in after running on the real data)
- Class balance: ...
- Any categories that look mislabeled or noisy: ...
- Job corpus quality issues found (duplicate postings, missing skills column, etc.): ...
- Decisions this informs for Phase 2 (recommender/clustering): ...